# Group analysis of columns data

## Load data

In [ ]:
import os

### This is to analyse the data in group
import numpy as np
import pandas as pd
from numexpr.necompiler import default_type

# Import columns data
csv_dir = r'/Volumes/newJetStor/newJetStor/paros/paros_WORK/hanwen/ad_decode_test/output/'
subject_list = r'/Volumes/newJetStor/newJetStor/paros/paros_WORK/hanwen/ad_decode_test/code/subjects_1mm.txt'
subject_group = r'/Users/bass/Desktop/HanwenLin/AD_DECODE_data3.xlsx'

# Read subject_list to get the subjects indexes
subjects = []
with open(subject_list, 'r') as f:
    for line in f:
        line = line.strip('\n')
        subjects.append(line)

print(subjects)

# Read subject_group to get subjects groups
ad_decode_data2 = pd.read_excel(subject_group, sheet_name='AD_DECODE_data2')
print('Size of the group data:', ad_decode_data2.shape)
print('Columns of the group data:', ad_decode_data2.columns)

In [ ]:
# Classified as left and right, risk 0 1 and 2 3, plot individual and averaged curves
# 4 categories in total
import matplotlib.pyplot as plt
import os

region_list = [
    'BA1_exvivo_thresh', 'BA1_exvivo',
    'BA2_exvivo_thresh', 'BA2_exvivo',
    'BA3a_exvivo_thresh', 'BA3a_exvivo',
    'BA3b_exvivo_thresh', 'BA3b_exvivo',
    'BA4a_exvivo_thresh', 'BA4a_exvivo',
    'BA4p_exvivo_thresh', 'BA4p_exvivo',
    'BA6_exvivo_thresh', 'BA6_exvivo',
    'BA44_exvivo_thresh', 'BA44_exvivo',
    'BA45_exvivo_thresh', 'BA45_exvivo',
    'cortex', 'cortex+hipamyg',
    'entorhinal_exvivo_thresh', 'entorhinal_exvivo',
    'FG1_mpm_vpnl', 'FG2_mpm_vpnl',
    'FG3_mpm_vpnl', 'FG4_mpm_vpnl',
    'hOc1_mpm_vpnl', 'hOc2_mpm_vpnl',
    'hOc3v_mpm_vpnl', 'hOc4v_mpm_vpnl',
    'MT_exvivo_thresh', 'MT_exvivo',
    'nofix_cortex',
    'perirhinal_exvivo_thresh', 'perirhinal_exvivo',
    'V1_exvivo_thresh', 'V1_exvivo',
    'V2_exvivo_thresh', 'V2_exvivo'
]
zero_risk_sub = ad_decode_data2[ad_decode_data2['risk_for_ad'] == 0]['MRI_Exam'].tolist()
one_risk_sub = ad_decode_data2[ad_decode_data2['risk_for_ad'] == 1]['MRI_Exam'].tolist()
two_risk_sub = ad_decode_data2[ad_decode_data2['risk_for_ad'] == 2]['MRI_Exam'].tolist()
three_risk_sub = ad_decode_data2[ad_decode_data2['risk_for_ad'] == 3]['MRI_Exam'].tolist()
low_risk_sub = zero_risk_sub + one_risk_sub
high_risk_sub = two_risk_sub + three_risk_sub
print(len(low_risk_sub))
print(len(high_risk_sub))


In [ ]:
left_low_dic = {region: np.empty([1, 21]) for region in region_list}
right_low_dic = {region: np.empty([1, 21]) for region in region_list}
rl_low_dic = {region: np.empty([1, 21]) for region in region_list}
left_high_dic = {region: np.empty([1, 21]) for region in region_list}
right_high_dic = {region: np.empty([1, 21]) for region in region_list}
rl_high_dic = {region: np.empty([1, 21]) for region in region_list}

def get_subject_id(s):
    integer = int(s)
    expected_length = 5
    length = len(str(integer))
    s_id = 'S'
    for i in range(expected_length - length):
        s_id = s_id + '0'
    s_id = s_id + str(integer)
    return s_id

for subject in high_risk_sub:
    subject_id = get_subject_id(subject)
    for region in region_list:
        lh_region_name = subject_id + '_lh_' + region + '_ad.csv'
        rh_region_name = subject_id + '_rh_' + region + '_ad.csv'
        rl_region_name = subject_id + '_rl_' + region + '_ad.csv'
        lh_csv = csv_dir + subject_id + '/columns_1mm/'+ lh_region_name
        rh_csv = csv_dir + subject_id + '/columns_1mm/'+ rh_region_name
        rl_csv = csv_dir + subject_id + '/columns_1mm/'+ rl_region_name
        if not os.path.exists(lh_csv):
            continue
        with open(lh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                continue
            left_high_dic[region] = np.append(left_high_dic[region], column.to_numpy(), axis=0)
        with open(rh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                continue
            right_high_dic[region] = np.append(right_high_dic[region], column.to_numpy(), axis=0)
        with open(rl_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                continue
            rl_high_dic[region] = np.append(rl_high_dic[region], column.to_numpy(), axis=0)

for subject in low_risk_sub:
    subject_id = get_subject_id(subject)
    for region in region_list:
        lh_region_name = subject_id + '_lh_' + region + '_ad.csv'
        rh_region_name = subject_id + '_rh_' + region + '_ad.csv'
        rl_region_name = subject_id + '_rl_' + region + '_ad.csv'
        lh_csv = csv_dir + subject_id + '/columns_1mm/'+ lh_region_name
        rh_csv = csv_dir + subject_id + '/columns_1mm/'+ rh_region_name
        rl_csv = csv_dir + subject_id + '/columns_1mm/'+ rl_region_name
        # Check if file exit
        if not os.path.exists(lh_csv):
            continue
        with open(lh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            # Check if column contains NaN
            if column.isna().any().any():
                continue
            left_low_dic[region] = np.append(left_low_dic[region], column.to_numpy(), axis=0)
        with open(rh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            # Check if column contains NaN
            if column.isna().any().any():
                continue
            right_low_dic[region] = np.append(right_low_dic[region], column.to_numpy(), axis=0)
        with open(rl_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                continue
            rl_low_dic[region] = np.append(rl_low_dic[region], column.to_numpy(), axis=0)

for subject in high_risk_sub:
    subject_id = get_subject_id(subject)
    for region in region_list:
        lh_region_name = subject_id + '_lh_' + region + '_ad.csv'
        rh_region_name = subject_id + '_rh_' + region + '_ad.csv'
        rl_region_name = subject_id + '_rl_' + region + '_ad.csv'
        lh_csv = csv_dir + subject_id + '/columns_1mm/'+ lh_region_name
        rh_csv = csv_dir + subject_id + '/columns_1mm/'+ rh_region_name
        rl_csv = csv_dir + subject_id + '/columns_1mm/'+ rl_region_name
        if not os.path.exists(lh_csv):
            continue
        with open(lh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                continue
            left_high_dic[region] = np.append(left_high_dic[region], column.to_numpy(), axis=0)
        with open(rh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                continue
            right_high_dic[region] = np.append(right_high_dic[region], column.to_numpy(), axis=0)
        with open(rl_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                continue
            rl_high_dic[region] = np.append(rl_high_dic[region], column.to_numpy(), axis=0)

for region in region_list:
    left_low_dic[region] = left_low_dic[region][1:]
    right_low_dic[region] = right_low_dic[region][1:]
    rl_low_dic[region] = rl_low_dic[region][1:]
    left_high_dic[region] = left_high_dic[region][1:]
    right_high_dic[region] = right_high_dic[region][1:]
    rl_high_dic[region] = rl_high_dic[region][1:]

In [ ]:
# Plot the columns
from matplotlib.lines import Line2D
for region in region_list:
    plt.figure()
    for i in range(left_low_dic[region].shape[0]):
        plt.plot(range(21), left_low_dic[region][i], color='b', linewidth=0.1)
    for i in range(right_low_dic[region].shape[0]):
        plt.plot(range(21), right_low_dic[region][i], color='r', linewidth=0.1)
    for i in range(left_high_dic[region].shape[0]):
        plt.plot(range(21), left_high_dic[region][i], color='g', linewidth=0.1)
    for i in range(right_high_dic[region].shape[0]):
        plt.plot(range(21), right_high_dic[region][i], color='c', linewidth=0.1)
    plt.plot(range(21), np.mean(left_low_dic[region], axis=0), color='b', linewidth=2)
    plt.plot(range(21), np.mean(right_low_dic[region], axis=0), color='r', linewidth=2)
    plt.plot(range(21), np.mean(left_high_dic[region], axis=0), color='g', linewidth=2)
    plt.plot(range(21), np.mean(right_high_dic[region], axis=0), color='c', linewidth=2)
    legend_elements = [Line2D([0], [0], color='b', linewidth=0.1, label='left_low_individual'),
                       Line2D([0], [0], color='r', linewidth=0.1, label='right_low_individual'),
                       Line2D([0], [0], color='g', linewidth=0.1, label='left_high_individual'),
                       Line2D([0], [0], color='c', linewidth=0.1, label='right_high_individual'),
                       Line2D([0], [0], color='b', linewidth=2, label='left_low_mean'),
                       Line2D([0], [0], color='r', linewidth=2, label='right_low_mean'),
                       Line2D([0], [0], color='g', linewidth=2, label='left_high_mean'),
                       Line2D([0], [0], color='c', linewidth=2, label='right_high_mean')]
    plt.legend(handles=legend_elements)
    plt.title(region)
    plt.xticks([0, 21], ['WM/GM','Pial'])
    plt.ylabel('AD')
    fig_dir = r'/Volumes/newJetStor/newJetStor/paros/paros_WORK/hanwen/ad_decode_test/output/linear_model_analysis/'
    plt.savefig(fig_dir + 'rd_' + region + '_1mm.png')
    plt.close()


In [ ]:
# Linear Model
valid_subjects = []
left_dic = {region: np.empty([1, 3]) for region in region_list}
right_dic = {region: np.empty([1, 3]) for region in region_list}
for subject in subjects:
    subject_id = subject
    file_valid = 'True'
    for region in region_list:
        lh_region_name = subject_id + '_lh_' + region + '_ad.csv'
        rh_region_name = subject_id + '_rh_' + region + '_ad.csv'
        lh_csv = csv_dir + subject_id + '/columns_1mm/'+ lh_region_name
        rh_csv = csv_dir + subject_id + '/columns_1mm/'+ rh_region_name
        if not os.path.exists(lh_csv):
            file_valid = 'False'
            print('Subject %s doesnt have lh_csv' % subject_id)
            continue
        if not os.path.exists(rh_csv):
            file_valid = 'False'
            print('Subject %s doesnt have rh_csv' % subject_id)
            continue
        with open(lh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                file_valid = 'False'
                print('NAN found in subject %s lh region %s', subject_id, region)
                continue
            three_bin = column.to_numpy()
            three_bin = np.array([[np.mean(three_bin[0, 0:7]), np.mean(three_bin[0, 7:14]), np.mean(three_bin[0, 14:])]])
            left_dic[region] = np.append(left_dic[region], three_bin, axis=0)
        with open(rh_csv, 'r') as f:
            column = pd.read_csv(f, dtype='float64', header=None)
            if column.isna().any().any():
                file_valid = 'False'
                print('NAN found in subject %s rh region %s', subject_id, region)
                continue
            three_bin = column.to_numpy()
            three_bin = np.array([[np.mean(three_bin[0, 0:7]), np.mean(three_bin[0, 7:14]), np.mean(three_bin[0, 14:])]])
            right_dic[region] = np.append(right_dic[region], three_bin, axis=0)
    if file_valid == 'True':
        valid_subjects.append(subject_id)

In [ ]:
valid_subject_index = []
for subject in valid_subjects:
    subject = subject[1:]
    subject_index = int(subject)
    valid_subject_index.append(subject_index)
for region in region_list:
    right_dic[region] = right_dic[region][1:]
    left_dic[region] = left_dic[region][1:]

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Construct X
X = np.empty([0,3])
age_column = np.empty([0,1])
genotype_column = np.empty([0,1])
sex_column = np.empty([0,1])
# Collect features: age, genotype and sex
for subject in valid_subject_index:
    age = ad_decode_data2[ad_decode_data2['MRI_Exam'] == subject]['age'].values[0]
    genotype = ad_decode_data2[ad_decode_data2['MRI_Exam'] == subject]['genotype'].values[0]
    sex = ad_decode_data2[ad_decode_data2['MRI_Exam'] == subject]['sex'].values[0]
    if '4' in genotype:
        genotype = 1
    else:
        genotype = 0
    if sex == 'M':
        sex = 1
    else:
        sex = 0
    age_column = np.append(age_column, age)
    genotype_column = np.append(genotype_column, genotype)
    sex_column = np.append(sex_column, sex)

age_column = age_column.reshape((-1, 1))
genotype_column = genotype_column.reshape((-1, 1))
sex_column = sex_column.reshape((-1, 1))
# Standard age
scaler = StandardScaler()
age_std = scaler.fit_transform(age_column)

# Standard genotype
mu_genotype = np.mean(genotype_column)
sigma_genotype = np.std(genotype_column)
genotype_std = (genotype_column - mu_genotype) / sigma_genotype

# Standard sex
mu_sex = np.mean(sex_column)
sigma_sex = np.std(sex_column)
sex_std = (sex_column - mu_sex) / sigma_sex

# Combine features
X = np.column_stack((age_std, genotype_std, sex_std))
print(X)

In [ ]:
from sklearn.metrics import r2_score
# For left hemisphere
for region in region_list:
    print('lh_' + region)
    Y = left_dic[region]
    linear_model = LinearRegression()
    linear_model.fit(X, Y)
    r2 = linear_model.score(X, Y)
    weights = linear_model.coef_
    Y_pred = linear_model.predict(X)
    mse = np.mean((Y_pred - Y) ** 2)
    print('Weights:')
    print(weights)
    print('MSE:')
    print(mse)
    print('R2:')
    print(r2)
    plt.subplot(2, 3, 1)
    plt.scatter(Y[:,0], X[:,0], label='age', color='r', linewidths=0.2)
    plt.scatter(Y[:,0], X[:,1], label='genotype', color='b', linewidths=0.2)
    plt.scatter(Y[:,0], X[:,2], label='sex', color='g', linewidths=0.2)
    plt.legend()
    plt.subplot(2, 3, 4)
    plt.scatter(Y[:,0], Y_pred[:,0], label='Y_pred', color='c')
    plt.plot(Y[:,0], Y[:,0], color='k')
    plt.legend()
    plt.xlabel('AD_bin1')
    plt.subplot(2, 3, 2)
    plt.scatter(Y[:,1], X[:,0], label='age', color='r', linewidths=0.2)
    plt.scatter(Y[:,1], X[:,1], label='genotype', color='b', linewidths=0.2)
    plt.scatter(Y[:,1], X[:,2], label='sex', color='g', linewidths=0.2)
    plt.legend()
    plt.subplot(2, 3, 5)
    plt.scatter(Y[:,1], Y_pred[:,1], label='Y_pred', color='c')
    plt.plot(Y[:,1], Y[:,1], color='k')
    plt.legend()
    plt.xlabel('AD_bin2')
    plt.subplot(2, 3, 3)
    plt.scatter(Y[:,2], X[:,0], label='age', color='r', linewidths=0.2)
    plt.scatter(Y[:,2], X[:,1], label='genotype', color='b', linewidths=0.2)
    plt.scatter(Y[:,2], X[:,2], label='sex', color='g', linewidths=0.2)
    plt.legend()
    plt.subplot(2, 3, 6)
    plt.scatter(Y[:,2], Y_pred[:,2], label='Y_pred', color='c')
    plt.plot(Y[:,2], Y[:,2], color='k')
    plt.legend()
    plt.xlabel('AD_bin3')
    plt.show()



In [ ]:
# Construct X
X = np.empty([0,3])
age_column = np.empty([0,1])
# Collect features: age, genotype and sex
for subject in valid_subject_index:
    age = ad_decode_data2[ad_decode_data2['MRI_Exam'] == subject]['age'].values[0]
    age_column = np.append(age_column, age)

age_column = age_column.reshape((-1, 1))
# Standard age
scaler = StandardScaler()
age_std = scaler.fit_transform(age_column)

# Combine features
X = age_column
print(X)

In [ ]:
# For left hemisphere
for region in region_list:
    print('lh_' + region)
    Y_1 = left_dic[region][:,0].reshape((-1, 1))
    linear_model = LinearRegression()
    linear_model.fit(X, Y_1)
    r2_1 = linear_model.score(X, Y_1)
    weights_1 = linear_model.coef_
    Y_pred_1 = linear_model.predict(X)
    mse_1 = np.mean((Y_pred_1 - Y_1) ** 2)

    Y_2 = left_dic[region][:,1].reshape((-1, 1))
    linear_model = LinearRegression()
    linear_model.fit(X, Y_2)
    r2_2 = linear_model.score(X, Y_2)
    weights_2 = linear_model.coef_
    Y_pred_2 = linear_model.predict(X)
    mse_2 = np.mean((Y_pred_2 - Y_2) ** 2)

    Y_3 = left_dic[region][:,2].reshape((-1, 1))
    linear_model = LinearRegression()
    linear_model.fit(X, Y_3)
    r2_3 = linear_model.score(X, Y_3)
    weights_3 = linear_model.coef_
    Y_pred_3 = linear_model.predict(X)
    mse_3 = np.mean((Y_pred_3 - Y_3) ** 2)

    print('Weights:')
    print(weights_1, weights_2, weights_3)
    print('MSE:')
    print(mse_1, mse_2, mse_3)
    print('R2:')
    print(r2_1, r2_2, r2_3)
    plt.subplot(3, 2, 1)
    plt.scatter(X[:,0], Y_1[:,0], label='Y_1', color='r', linewidths=0.2)
    plt.plot(X[:,0], Y_pred_1[:,0], label='Y_pred_1', color='b')
    plt.legend()
    plt.xlabel('age')
    plt.subplot(3, 2, 2)
    plt.scatter(Y_1[:,0], Y_pred_1[:,0], label='Y_pred_1', color='c')
    plt.plot(Y_1[:,0], Y_1[:,0], color='k')
    plt.legend()
    plt.xlabel('AD')
    plt.subplot(3, 2, 3)
    plt.scatter(X[:,0], Y_2[:,0], label='Y_2', color='r', linewidths=0.2)
    plt.plot(X[:,0], Y_pred_2[:,0], label='Y_pred_2', color='b')
    plt.legend()
    plt.xlabel('age')
    plt.subplot(3, 2, 4)
    plt.scatter(Y_2[:,0], Y_pred_2[:,0], label='Y_pred_2', color='c')
    plt.plot(Y_2[:,0], Y_2[:,0], color='k')
    plt.legend()
    plt.xlabel('AD')
    plt.subplot(3, 2, 5)
    plt.scatter(X[:,0], Y_3[:,0], label='Y_3', color='r', linewidths=0.2)
    plt.plot(X[:,0], Y_pred_3[:,0], label='Y_pred_3', color='b')
    plt.legend()
    plt.xlabel('age')
    plt.subplot(3, 2, 6)
    plt.scatter(Y_3[:,0], Y_pred_3[:,0], label='Y_pred_3', color='c')
    plt.plot(Y_3[:,0], Y_3[:,0], color='k')
    plt.legend()
    plt.xlabel('AD')

    plt.show()


In [ ]:
# For left hemisphere
for region in region_list:
    print('rh_' + region)
    Y_1 = right_dic[region][:,0].reshape((-1, 1))
    linear_model = LinearRegression()
    linear_model.fit(X, Y_1)
    r2_1 = linear_model.score(X, Y_1)
    weights_1 = linear_model.coef_
    Y_pred_1 = linear_model.predict(X)
    mse_1 = np.mean((Y_pred_1 - Y_1) ** 2)

    Y_2 = right_dic[region][:,1].reshape((-1, 1))
    linear_model = LinearRegression()
    linear_model.fit(X, Y_2)
    r2_2 = linear_model.score(X, Y_2)
    weights_2 = linear_model.coef_
    Y_pred_2 = linear_model.predict(X)
    mse_2 = np.mean((Y_pred_2 - Y_2) ** 2)

    Y_3 = right_dic[region][:,2].reshape((-1, 1))
    linear_model = LinearRegression()
    linear_model.fit(X, Y_3)
    r2_3 = linear_model.score(X, Y_3)
    weights_3 = linear_model.coef_
    Y_pred_3 = linear_model.predict(X)
    mse_3 = np.mean((Y_pred_3 - Y_3) ** 2)

    print('Weights:')
    print(weights_1, weights_2, weights_3)
    print('MSE:')
    print(mse_1, mse_2, mse_3)
    print('R2:')
    print(r2_1, r2_2, r2_3)
    plt.subplot(3, 2, 1)
    plt.scatter(X[:,0], Y_1[:,0], label='Y_1', color='r', linewidths=0.2)
    plt.plot(X[:,0], Y_pred_1[:,0], label='Y_pred_1', color='b')
    plt.legend()
    plt.xlabel('age')
    plt.subplot(3, 2, 2)
    plt.scatter(Y_1[:,0], Y_pred_1[:,0], label='Y_pred_1', color='c')
    plt.plot(Y_1[:,0], Y_1[:,0], color='k')
    plt.legend()
    plt.xlabel('AD')
    plt.subplot(3, 2, 3)
    plt.scatter(X[:,0], Y_2[:,0], label='Y_2', color='r', linewidths=0.2)
    plt.plot(X[:,0], Y_pred_2[:,0], label='Y_pred_2', color='b')
    plt.legend()
    plt.xlabel('age')
    plt.subplot(3, 2, 4)
    plt.scatter(Y_2[:,0], Y_pred_2[:,0], label='Y_pred_2', color='c')
    plt.plot(Y_2[:,0], Y_2[:,0], color='k')
    plt.legend()
    plt.xlabel('AD')
    plt.subplot(3, 2, 5)
    plt.scatter(X[:,0], Y_3[:,0], label='Y_3', color='r', linewidths=0.2)
    plt.plot(X[:,0], Y_pred_3[:,0], label='Y_pred_3', color='b')
    plt.legend()
    plt.xlabel('age')
    plt.subplot(3, 2, 6)
    plt.scatter(Y_3[:,0], Y_pred_3[:,0], label='Y_pred_3', color='c')
    plt.plot(Y_3[:,0], Y_3[:,0], color='k')
    plt.legend()
    plt.xlabel('AD')

    plt.show()
